In [1]:
%load_ext autoreload
%autoreload 2

# WW3 boundaries: synthetic reference data, end to end

Sets up a small CrocoDash case with WW3 as the **active** wave component
(`...SGLC_WW3_SESP` instead of the stub `SWAV`) and runs it through
`configure_forcings()`/`process_forcings()` using the fast, deterministic
**`reference_ocean`**/**`reference_waves`** products for MOM6's IC/OBC and
WW3's own OBC respectively -- no CDS API key, no campaign-storage access,
no network at all. There's no separate manual copy-and-slice step and no
placeholder-then-real two-pass dance -- one `configure_forcings()` call,
one `process_forcings()` call, same as every other component (MOM6, CICE,
...).

Every synthetic grid point in each boundary's buffered window becomes its
own station (see `ww3.py`'s `_regrid_chunk_era5`/`_extract_all_stations`
-- `reference_waves` reuses the same multi-station code path as real ERA5),
not collapsed to one point -- `ww3_bounc.nml`'s `INTERP` is chosen
accordingly (`2`/linear for multi-station, vs. `1`/nearest for the
placeholder `WW3` product's one-station-per-boundary case).

To swap in the real, slow ERA5 product instead, pass
`ww3_obc_product_name="era5_wave_spectra"`,
`ww3_obc_function_name="get_era5_2d_spectra"` to `configure_forcings` below
(see `get_era5_2d_spectra`'s own description in
`raw_data_access/datasets/era5.py` for CDS key setup).

## Step 0: Paths and imports

Follows the "Fast CrocoDash Case Setup" pattern from the repo's `CLAUDE.md`
-- same small flat-bathymetry domain used there, just with WW3 swapped in.


In [2]:
from pathlib import Path

from CrocoDash.grid import Grid
from CrocoDash.topo import Topo
from CrocoDash.vgrid import VGrid
from CrocoDash.case import Case

CESMROOT = Path("~/work/installs/full_regional_cesm").expanduser()
CASENAME = "ww3_boundaries_demo_manual"
CASEROOT = (Path("~/croc_cases") / CASENAME).expanduser()
INPUTDIR = (Path("~/scratch/croc_input") / CASENAME).expanduser()

print(CESMROOT)
print(CASEROOT)
print(INPUTDIR)

/glade/u/home/manishrv/work/installs/full_regional_cesm
/glade/u/home/manishrv/croc_cases/ww3_boundaries_demo_manual
/glade/u/home/manishrv/scratch/croc_input/ww3_boundaries_demo_manual


## Step 1: Grid, topography, vertical grid

The exact domain from CLAUDE.md's fast-setup example (10x10 degrees at
1-degree resolution, flat 1000m bathymetry, uniform vgrid) -- this is a
wiring demo, not a science case, so there's no reason to use anything
fancier.

In [3]:
grid = Grid(
    lenx=10.0,
    leny=10.0,
    resolution=1.0,
    xstart=-60.0,
    ystart=30.0,
    name=CASENAME,
)
topo = Topo(grid, min_depth=10.0, git=False)
topo.set_flat(1000.0)
vgrid = VGrid.uniform(nk=10, depth=1000.0, name=CASENAME)

## Step 2: Create the case

Just `SWAV` -> `WW3_SESP` in the NYF compset to get the real coupled wave
model instead of the stub. (`SESP` is CESM's stub "external system
processing" component, unrelated to waves -- it's just what this compset
alias always appends after an active WAV component.)

In [4]:
case = Case(
    cesmroot=CESMROOT,
    caseroot=CASEROOT,
    inputdir=INPUTDIR,
    compset="1850_DATM%NYF_SLND_SICE_MOM6%REGIONAL_SROF_SGLC_WW3_SESP",
    ocn_grid=grid,
    ocn_topo=topo,
    ocn_vgrid=vgrid,
    atm_grid_name="T62",
    machine="derecho",
    project="ncgd0011",
    override=True,
)
print("WW3 in compset:", case.ww3_in_compset)

INFO:	initialize:Initializing the visualCaseGen system...


INFO:  csp_solver:CspSolver initialized.


Creating case...

Running the create_newcase tool with the following command:

/glade/u/home/manishrv/work/installs/full_regional_cesm/cime/scripts/create_newcase --compset 1850_DATM%NYF_SLND_SICE_MOM6%REGIONAL_SROF_SGLC_WW3_SESP --res USER_RES --case /glade/u/home/manishrv/croc_cases/ww3_boundaries_demo_manual --machine derecho --run-unsupported --project ncgd0011 --non-local 



The create_newcase command was successful.

• Updating case xml variables to include newly generated ocean grid "ww3_boundaries_demo_manual" with the following properties:
 nx: 10, ny: 10. ocean mesh: /glade/u/home/manishrv/scratch/croc_input/ww3_boundaries_demo_manual/ocean/ESMF_mesh_ww3_boundaries_demo_manual_286d35.nc.

./xmlchange OCN_NX=10 --non-local



./xmlchange OCN_NY=10 --non-local



./xmlchange OCN_DOMAIN_MESH=/glade/u/home/manishrv/scratch/croc_input/ww3_boundaries_demo_manual/ocean/ESMF_mesh_ww3_boundaries_demo_manual_286d35.nc --non-local



... (14 outputs left out)

Adding parameter changes to user_nl_mom:

  ! Custom Horizonal Grid, Topography, and Vertical Grid
  INPUTDIR = /glade/u/home/manishrv/scratch/croc_input/ww3_boundaries_demo_manual/ocean
  TRIPOLAR_N = False
  REENTRANT_X = False
  REENTRANT_Y = False
  NIGLOBAL = 10
  NJGLOBAL = 10
  GRID_CONFIG = mosaic
  GRID_FILE = ocean_hgrid_ww3_boundaries_demo_manual_286d35.nc
  TOPO_CONFIG = file
  TOPO_FILE = ocean_topog_ww3_boundaries_demo_manual_286d35.nc
  MAXIMUM_DEPTH = 1000.0
  MINIMUM_DEPTH = 10.0
  NK = 10
  COORD_CONFIG = none
  ALE_COORDINATE_CONFIG = FILE:ocean_vgrid_ww3_boundaries_demo_manual_286d35.nc
  REGRIDDING_COORDINATE_MODE = Z*

  ! Timesteps (based on grid resolution)
  DT = 1800.0
  DT_THERM = 1800.0



INFO:	stage:SUCCESS: All stages are complete.


Case created successfully at /glade/u/home/manishrv/croc_cases/ww3_boundaries_demo_manual.

To further customize, build, and run the case, navigate to the case directory in your terminal. To create another case, restart the notebook.

./xmlchange MOM6_MEMORY_MODE=dynamic_symmetric --non-local



The following additional configuration options are required to run and must be provided with any listed arguments in configure_forcings:
  - WW3: boundaries
  - conditions: boundaries, product_name, function_name, compset
WW3 in compset: True


## Step 3: Configure + process forcings

`configure_forcings` sets up MOM6's IC/OBC (`product_name`/`function_name`,
pointed at `reference_ocean`) *and*, since WW3 is active, the WW3 OBC
configurator (`ww3_obc_product_name`/`ww3_obc_function_name`, pointed at
`reference_waves`). `process_forcings()` then runs the driver workflow for
everything active for this compset: MOM6 IC, MOM6 OBC, and WW3 OBC --
entirely from fast, deterministic synthetic data, in seconds.

In [5]:
case.configure_forcings(
    date_range=["2020-01-01 00:00:00", "2020-01-09 00:00:00"],
    product_name="reference_ocean",
    function_name="get_reference_ocean_data",
    ww3_obc_product_name="reference_waves",
    ww3_obc_function_name="get_reference_wave_spectra",
)

INFO:CrocoDash.forcing_configurations.base:[SKIP] tides missing args: ['tpxo_elevation_filepath', 'tpxo_velocity_filepath', 'tidal_constituents']


INFO:CrocoDash.forcing_configurations.base:[SKIP] BGC incompatible with compset


INFO:CrocoDash.forcing_configurations.base:[REQUIRED] Activating WW3


INFO:CrocoDash.forcing_configurations.base:[SKIP] CICE incompatible with compset


INFO:CrocoDash.forcing_configurations.base:[SKIP] BGCIC incompatible with compset


INFO:CrocoDash.forcing_configurations.base:[SKIP] BGCIronForcing incompatible with compset


... (8 outputs left out)

INFO:CrocoDash.forcing_configurations.base:Configuring conditions


Adding parameter changes to user_nl_mom:

  ! Initial conditions
  INIT_LAYERS_FROM_Z_FILE = True
  Z_INIT_ALE_REMAPPING = True
  TEMP_SALT_INIT_VERTICAL_REMAP_ONLY = True
  DEPRESS_INITIAL_SURFACE = True
  VELOCITY_CONFIG = file
  TEMP_SALT_Z_INIT_FILE = init_tracers.nc
  SURFACE_HEIGHT_IC_FILE = init_eta.nc
  VELOCITY_FILE = init_vel.nc
  Z_INIT_FILE_PTEMP_VAR = temp
  Z_INIT_FILE_SALT_VAR = salt
  SURFACE_HEIGHT_IC_VAR = eta_t
  U_IC_VAR = u
  V_IC_VAR = v

  ! Open boundary conditions
  OBC_NUMBER_OF_SEGMENTS = 4
  OBC_FREESLIP_VORTICITY = False
  OBC_FREESLIP_STRAIN = False
  OBC_COMPUTED_VORTICITY = True
  OBC_COMPUTED_STRAIN = True
  OBC_ZERO_BIHARMONIC = True
  OBC_TRACER_RESERVOIR_LENGTH_SCALE_OUT = 3.0E+04
  OBC_TRACER_RESERVOIR_LENGTH_SCALE_IN = 3000.0
  BRUSHCUTTER_MODE = True
  OBC_SEGMENT_001 = "J=0,I=0:N,FLATHER,ORLANSKI,NUDGED,ORLANSKI_TAN,NUDGED_TAN"
  OBC_SEGMENT_001_VELOCITY_NUDGING_TIMESCALES = 0.3, 360.0
  OBC_SEGMENT_001_DATA = "U=file:forcing_obc_segment_001.nc(u

./xmlchange STOP_OPTION=ndays --non-local



./xmlchange STOP_N=8 --non-local



In [6]:
case.process_forcings()

INFO:CrocoDash.extract_forcings.obc:GET [south]: 2020-01-01 → 2020-01-09


INFO:CrocoDash.extract_forcings.obc:GET [north]: 2020-01-01 → 2020-01-09


INFO:CrocoDash.extract_forcings.obc:GET [west]: 2020-01-01 → 2020-01-09


INFO:CrocoDash.extract_forcings.obc:GET [east]: 2020-01-01 → 2020-01-09


INFO:CrocoDash.extract_forcings.obc:REGRID [south]: 9-day slices


No arakawa_grid provided, assuming the variable mapping for your data product is already in correct format.
Arakawa A grid detected in variable mapping


... (42 outputs left out)

INFO:CrocoDash.extract_forcings.obc:MERGE [east]


INFO:CrocoDash.extract_forcings.obc:Saved merged boundary at /glade/u/home/manishrv/scratch/croc_input/ww3_boundaries_demo_manual/extract_forcings/ww3/merged/forcing_obc_segment_004.nc


INFO:CrocoDash.extract_forcings.obc:OBC processing complete.


[timing] bc: 24.7s  ic: 1.1s  ww3: 1.6s  total: 27.3s
Case is ready to be built: /glade/u/home/manishrv/croc_cases/ww3_boundaries_demo_manual


## Inspect the synthetic multi-station reference output

Every synthetic reference grid point in each boundary's buffered window is
its own station -- hundreds across the domain's perimeter, not one point
per boundary -- each carrying its own JONSWAP-shaped `E(f, theta)`
spectrum, left for `ww3_bounc`'s linear interpolation (`BOUND%INTERP = 2`)
to blend between.

In [7]:
wave_dir = INPUTDIR / "wave"

spec_names = (wave_dir / "spec.list").read_text().splitlines()
print("total synthetic stations across all 4 boundaries:", len(spec_names))
print(
    [l for l in (wave_dir / "ww3_bounc.nml").read_text().splitlines() if "INTERP" in l][-1]
)

total synthetic stations across all 4 boundaries: 12
  BOUND%INTERP               = 2


In [8]:
import xarray as xr

lats, lons, maxes, nonzero = [], [], [], 0
for name in spec_names:
    ds = xr.open_dataset(wave_dir / name, decode_times=False)
    lats.append(float(ds["latitude"].values[0]))
    lons.append(float(ds["longitude"].values[0]))
    maxes.append(float(ds["efth"].max()))
    if (ds["efth"].values != 0).any():
        nonzero += 1
    ds.close()

print(f"stations with nonzero wave energy: {nonzero}/{len(spec_names)}")
print(f"latitude range:  {min(lats):.2f} to {max(lats):.2f}")
print(f"longitude range: {min(lons):.2f} to {max(lons):.2f}")
print(f"peak efth across all stations: {max(maxes):.3f} m^2 s / rad")

stations with nonzero wave energy: 12/12
latitude range:  30.00 to 40.00
longitude range: -60.00 to -50.00
peak efth across all stations: 11.061 m^2 s / rad


## Case is set up

`./case.build`/`./case.submit` are intentionally not run here -- this
notebook is about validating the OBC-generation pipeline, not running the
coupled model. That's a natural next step once this looks right.

In [9]:
print("caseroot:", case.caseroot)
print("inputdir:", case.inputdir)

caseroot: /glade/u/home/manishrv/croc_cases/ww3_boundaries_demo_manual
inputdir: /glade/u/home/manishrv/scratch/croc_input/ww3_boundaries_demo_manual
